# 10 Property Advisory with Local Listing Context

Use this notebook to test property advisory after loading the expanded seed listings and rebuilding the knowledge index. This flow combines price prediction, retrieved knowledge, and nearby demo listing context.

In [1]:
import os
import sys

sys.path.append(os.path.abspath('..'))

In [2]:
from src.data.load_property_listings import main as load_property_listings_main
from src.db.session import create_db_tables
from src.rag.build_index import main as build_index_main
from src.rag.service import ask_property_question

/Volumes/NIKHILESH/Projects/real-estate-ai-advisor/real-estate-ai-platform/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
create_db_tables()
load_property_listings_main()
build_index_main()

Loaded 30 property listings from /Volumes/NIKHILESH/Projects/real-estate-ai-advisor/real-estate-ai-platform/data/sample/property_listings_seed.csv
Saved FAISS index to /Volumes/NIKHILESH/Projects/real-estate-ai-advisor/real-estate-ai-platform/data/knowledge/index/knowledge.index
Saved chunk metadata to /Volumes/NIKHILESH/Projects/real-estate-ai-advisor/real-estate-ai-platform/data/knowledge/index/chunks.json
Saved index metadata to /Volumes/NIKHILESH/Projects/real-estate-ai-advisor/real-estate-ai-platform/data/knowledge/index/index_metadata.json
Documents loaded: 5
Chunks created: 19


In [4]:
property_features = {
    'median_income': 8.3252,
    'house_age': 41.0,
    'average_rooms': 6.984127,
    'average_bedrooms': 1.02381,
    'population': 322.0,
    'average_occupancy': 2.555556,
    'latitude': 37.809,
    'longitude': -122.257,
}

In [5]:
result = ask_property_question(
    'How should I interpret this predicted price for a buyer who wants Oakland city access?',
    property_features,
)
result['model_name'], result['predicted_price'], result['predicted_price_usd']

('qwen2.5:14b', 3.7826945781707764, 378269.45781707764)

In [6]:
result['answer']

"1. The predicted price of $378,269 is below typical Oakland property prices based on the demo inventory.\n2. Practically, this model estimate suggests a relatively affordable option for buyers seeking access to Oakland's urban amenities and city lifestyle, particularly in less expensive neighborhoods or with smaller unit types like apartments or starter condos.\n3. According to the provided demo listing context, Oakland listings generally start around $600k for condos near desirable areas such as Lake Merritt. The predicted price is notably lower than these typical entry-level prices, indicating a potential bargain or an outlier in terms of size and features compared to nearby inventory.\n4. Note that this analysis relies on demo inventory data rather than actual market transactions, which may not fully represent current real estate conditions in Oakland."

In [7]:
[(item['title'], item['score']) for item in result['sources']]

[('Demo Locality Preference Notes', 0.7807029485702515),
 ('Demo Listing Inventory Snapshot', 0.7609350085258484),
 ('Demo Listing Inventory Snapshot', 0.7587953805923462)]